# Empirical Landscape Analytical Rho Bounds Table

This standalone notebook computes analytical squared-fitness decay metrics and manuscript bounds for the empirical landscapes used in Figure 6. It expands amino-acid landscapes into nucleotide/codon space, evaluates the uniform, weighted-undirected, and directed mutation operators, and saves a LaTeX table.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Latex, display
from matplotlib.figure import Figure

from slide.direvo_functions import CODON_MAPPER
from slide.utils import get_figures_dir, get_processed_data_dir

SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
DECIMALS: int = 4

PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
REPO_ROOT = Path.cwd()

LANDSCAPE_FILES: tuple[tuple[str, str], ...] = (
    ("GB1", "GB1_landscape_array.pkl"),
    ("TrpB", "TrpB_landscape_array.pkl"),
    ("TEV", "TEV_landscape_array.pkl"),
    ("ParD3", "E3_landscape_array.pkl"),
)

MODEL_ROWS: tuple[str, ...] = (
    r"Uniform ($\rho_2$)",
    r"$\mathit{E.\ coli}$ weighted ($\tilde{\rho}_2$)",
    r"Bound (17), $\mathit{E.\ coli}$ weighted",
    r"$\mathit{E.\ coli}$ directed ($\bar{\rho}_2$)",
    r"Bound (21), $\mathit{E.\ coli}$ directed",
    r"$\mathit{A.\ thaliana}$ weighted ($\tilde{\rho}_2$)",
    r"Bound (17), $\mathit{A.\ thaliana}$ weighted",
    r"$\mathit{A.\ thaliana}$ directed ($\bar{\rho}_2$)",
    r"Bound (21), $\mathit{A.\ thaliana}$ directed",
)

TABLE_PKL_PATH = PROCESSED_DATA_DIR / "empirical_landscape_rho_bounds_table.pkl"
TABLE_TEX_PATH = PROCESSED_DATA_DIR / "empirical_landscape_rho_bounds_table.tex"
FIGURE_STEM = "empirical_landscape_rho_bounds_table"

print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")


## Helper functions

In [ ]:
CODON_MAPPER_ARRAY = np.asarray(CODON_MAPPER, dtype=np.int16)
UNIFORM_KERNEL = (np.ones((4, 4), dtype=np.float64) - np.eye(4, dtype=np.float64)) / 3.0


def symmetric_sinkhorn_kernel(kernel: np.ndarray, tolerance: float = 1e-13) -> np.ndarray:
    """Create a symmetric doubly-stochastic nucleotide kernel.

    Parameters:
    - kernel: np.ndarray
        Non-negative row-stochastic nucleotide transition matrix.
    - tolerance: float
        Maximum permitted row-sum error before convergence.

    Returns:
    - np.ndarray
        Symmetric doubly-stochastic kernel preserving the input zero pattern.
    """
    symmetric = 0.5 * (np.asarray(kernel, dtype=np.float64) + np.asarray(kernel, dtype=np.float64).T)
    scale = np.ones(symmetric.shape[0], dtype=np.float64)
    for _ in range(100_000):
        row_sums = scale * (symmetric @ scale)
        if np.max(np.abs(row_sums - 1.0)) < tolerance:
            break
        scale *= np.sqrt(1.0 / row_sums)
    else:
        raise RuntimeError("Symmetric Sinkhorn scaling did not converge.")
    return scale[:, None] * symmetric * scale[None, :]


def stationary_distribution(kernel: np.ndarray) -> np.ndarray:
    """Return the normalized stationary distribution of a row-stochastic kernel.

    Parameters:
    - kernel: np.ndarray
        Row-stochastic transition matrix.

    Returns:
    - np.ndarray
        Positive stationary probability vector.
    """
    eigenvalues, eigenvectors = np.linalg.eig(np.asarray(kernel, dtype=np.float64).T)
    index = int(np.argmin(np.abs(eigenvalues - 1.0)))
    stationary = np.real(eigenvectors[:, index])
    if stationary.sum() < 0.0:
        stationary *= -1.0
    stationary /= stationary.sum()
    if np.any(stationary <= 0.0):
        raise ValueError("Stationary distribution contains non-positive probabilities.")
    return stationary


def load_landscape(filename: str) -> np.ndarray:
    """Load one empirical amino-acid landscape.

    Parameters:
    - filename: str
        Pickle filename inside ``landscape_arrays``.

    Returns:
    - np.ndarray
        Empirical amino-acid landscape.
    """
    with (REPO_ROOT / "landscape_arrays" / filename).open("rb") as handle:
        landscape = pickle.load(handle)
    return np.asarray(landscape, dtype=np.float32)


def build_nucleotide_landscape_chunked(landscape: np.ndarray, chunk_size: int = 1_000_000) -> np.ndarray:
    """Expand an amino-acid landscape into nucleotide/codon space in chunks.

    Parameters:
    - landscape: np.ndarray
        Amino-acid landscape with one axis per residue.
    - chunk_size: int
        Number of linear nucleotide genotypes decoded at once.

    Returns:
    - np.ndarray
        Nucleotide landscape with three four-state axes per residue.
    """
    amino_acid_landscape = np.asarray(landscape, dtype=np.float32)
    num_amino_acid_sites = amino_acid_landscape.ndim
    num_nucleotide_sites = 3 * num_amino_acid_sites
    total_genotypes = 4**num_nucleotide_sites
    minimum_fitness = float(np.nanmin(amino_acid_landscape))
    buffered = np.pad(
        amino_acid_landscape,
        [(0, 1)] * num_amino_acid_sites,
        constant_values=minimum_fitness,
    )
    flat_values = np.empty(total_genotypes, dtype=np.float32)
    for start in range(0, total_genotypes, chunk_size):
        stop = min(start + chunk_size, total_genotypes)
        linear_ids = np.arange(start, stop, dtype=np.uint64)
        amino_acid_indices: list[np.ndarray] = []
        for site in range(num_amino_acid_sites):
            shifts = [2 * (num_nucleotide_sites - 1 - (3 * site + nucleotide)) for nucleotide in range(3)]
            first = ((linear_ids >> np.uint64(shifts[0])) & np.uint64(3)).astype(np.int16)
            second = ((linear_ids >> np.uint64(shifts[1])) & np.uint64(3)).astype(np.int16)
            third = ((linear_ids >> np.uint64(shifts[2])) & np.uint64(3)).astype(np.int16)
            amino_acid_indices.append(CODON_MAPPER_ARRAY[first, second, third])
        flat_values[start:stop] = buffered[tuple(amino_acid_indices)]
    return flat_values.reshape((4,) * num_nucleotide_sites)


def stationary_average(values: np.ndarray, stationary: np.ndarray) -> float:
    """Compute a product-stationary landscape average.

    Parameters:
    - values: np.ndarray
        Nucleotide-space fitness landscape.
    - stationary: np.ndarray
        Single-site stationary distribution.

    Returns:
    - float
        Product-distribution weighted average.
    """
    contracted = np.asarray(values, dtype=np.float64)
    for _ in range(values.ndim):
        contracted = np.tensordot(stationary, contracted, axes=([0], [0]))
    return float(contracted)


def axis_quadratic(values: np.ndarray, kernel: np.ndarray, axis: int) -> float:
    """Compute ``f^T K_axis f`` without forming ``K_axis``.

    Parameters:
    - values: np.ndarray
        Nucleotide-space fitness landscape.
    - kernel: np.ndarray
        Single-site transition kernel.
    - axis: int
        Landscape axis receiving the kernel.

    Returns:
    - float
        Quadratic form for one axis.
    """
    moved = np.moveaxis(values, axis, 0).reshape(4, -1).astype(np.float64, copy=False)
    return float(np.einsum("ik,ij,jk->", moved, kernel, moved, optimize=True))


def analytical_squared_decay_from_stats(
    values: np.ndarray,
    kernel: np.ndarray,
    directed: bool,
    squared_norm: float,
) -> dict[str, float | np.ndarray]:
    """Compute analytical rho_2 with the Figure 6 convention.

    Parameters:
    - values: np.ndarray
        Nucleotide-space fitness landscape.
    - kernel: np.ndarray
        Single-site row-stochastic kernel.
    - directed: bool
        Whether to use the symmetrized directed Laplacian and stationary asymptote.
    - squared_norm: float
        Precomputed ``f^T f``.

    Returns:
    - dict[str, float | np.ndarray]
        Analytical rate, numerator, denominator, asymptote term, and stationary distribution.
    """
    stationary = stationary_distribution(kernel)
    mean = stationary_average(values, stationary) if directed else float(values.mean(dtype=np.float64))
    b_zero = float(values.size * mean * mean)
    denominator = float(squared_norm - b_zero)
    if denominator <= 0.0:
        raise ValueError("Squared-decay denominator must be positive.")
    kernel_sum = 0.0
    for axis in range(values.ndim):
        if directed:
            kernel_sum += 0.5 * (
                axis_quadratic(values, kernel, axis)
                + axis_quadratic(values, kernel.T, axis)
            )
        else:
            kernel_sum += axis_quadratic(values, kernel, axis)
    numerator = float(values.ndim * squared_norm - kernel_sum)
    rho_2 = float(numerator / (values.ndim * denominator))
    return {
        "rho_2": rho_2,
        "numerator": numerator,
        "denominator": denominator,
        "b_0": b_zero,
        "stationary_distribution": stationary,
    }


def single_site_operator_norm(kernel_difference: np.ndarray) -> float:
    """Compute a conservative per-site spectral norm for a kernel perturbation.

    Parameters:
    - kernel_difference: np.ndarray
        Difference between two single-site transition kernels.

    Returns:
    - float
        Matrix spectral norm.
    """
    return float(np.linalg.norm(np.asarray(kernel_difference, dtype=np.float64), ord=2))


def weighted_bound_17(
    squared_norm: float,
    uniform_denominator: float,
    weighted_kernel: np.ndarray,
) -> float:
    """Compute the Eq. 17 weighted-undirected bound in Figure 6 units.

    Parameters:
    - squared_norm: float
        ``f^T f`` for the nucleotide-expanded landscape.
    - uniform_denominator: float
        ``f^T f - b_0`` for the uniform asymptote.
    - weighted_kernel: np.ndarray
        Symmetric weighted-undirected single-site kernel.

    Returns:
    - float
        Upper bound on ``abs(rho_2 - rho_tilde_2)``.
    """
    return float(squared_norm * single_site_operator_norm(weighted_kernel - UNIFORM_KERNEL) / uniform_denominator)


def directed_bound_21(
    squared_norm: float,
    uniform_denominator: float,
    uniform_b_zero: float,
    directed_kernel: np.ndarray,
    directed_result: dict[str, float | np.ndarray],
    num_axes: int,
) -> float:
    """Compute the directed bound labelled Eq. 21 in the paper draft.

    Parameters:
    - squared_norm: float
        ``f^T f`` for the nucleotide-expanded landscape.
    - uniform_denominator: float
        ``f^T f - b_0`` for the uniform asymptote.
    - uniform_b_zero: float
        Uniform-asymptote ``b_0``.
    - directed_kernel: np.ndarray
        Directed row-stochastic single-site kernel.
    - directed_result: dict[str, float | np.ndarray]
        Analytical directed result containing numerator, denominator, and directed ``b_0``.
    - num_axes: int
        Number of nucleotide landscape axes used for Figure 6 normalization.

    Returns:
    - float
        Upper bound on ``abs(rho_2 - rho_bar_2)`` in Figure 6 units.
    """
    directed_symmetric_kernel = 0.5 * (directed_kernel + directed_kernel.T)
    first_term = squared_norm * single_site_operator_norm(directed_symmetric_kernel - UNIFORM_KERNEL) / uniform_denominator
    directed_denominator = float(directed_result["denominator"])
    directed_b_zero = float(directed_result["b_0"])
    directed_numerator = abs(float(directed_result["numerator"]))
    second_term = (
        directed_numerator
        * abs(uniform_b_zero - directed_b_zero)
        / (num_axes * uniform_denominator * directed_denominator)
    )
    return float(first_term + second_term)


def load_mutation_kernels() -> dict[str, np.ndarray]:
    """Load the mutation kernels used in Figure 6.

    Parameters:
    - None
        Kernels are loaded from ``other_data``.

    Returns:
    - dict[str, np.ndarray]
        Uniform, weighted, and directed nucleotide kernels.
    """
    e_coli_directed = np.asarray(np.load(REPO_ROOT / "other_data" / "normed_e_coli_matrix.npy"), dtype=np.float64)
    a_thaliana_directed = np.asarray(np.load(REPO_ROOT / "other_data" / "normed_a_thaliana_matrix.npy"), dtype=np.float64)
    kernels = {
        "uniform": UNIFORM_KERNEL,
        "e_coli_weighted": symmetric_sinkhorn_kernel(e_coli_directed),
        "e_coli_directed": e_coli_directed,
        "a_thaliana_weighted": symmetric_sinkhorn_kernel(a_thaliana_directed),
        "a_thaliana_directed": a_thaliana_directed,
    }
    for name, kernel in kernels.items():
        if kernel.shape != (4, 4):
            raise ValueError(f"{name} kernel has shape {kernel.shape}.")
        if np.any(kernel < 0.0) or not np.allclose(kernel.sum(axis=1), 1.0, atol=1e-10):
            raise ValueError(f"{name} kernel is not row-stochastic.")
    return kernels


## Compute analytical rates and bounds

In [ ]:
kernels = load_mutation_kernels()
numeric_rows: dict[str, dict[str, float]] = {row: {} for row in MODEL_ROWS}
diagnostics: dict[str, dict[str, float]] = {}

for landscape_name, filename in LANDSCAPE_FILES:
    print(f"Computing {landscape_name}...")
    amino_acid_landscape = load_landscape(filename)
    nucleotide_landscape = build_nucleotide_landscape_chunked(amino_acid_landscape)
    values_ndim = int(nucleotide_landscape.ndim)
    squared_norm = float(np.einsum("...,...->", nucleotide_landscape, nucleotide_landscape, optimize=True))

    uniform_result = analytical_squared_decay_from_stats(
        nucleotide_landscape,
        kernels["uniform"],
        directed=False,
        squared_norm=squared_norm,
    )
    e_coli_weighted_result = analytical_squared_decay_from_stats(
        nucleotide_landscape,
        kernels["e_coli_weighted"],
        directed=False,
        squared_norm=squared_norm,
    )
    e_coli_directed_result = analytical_squared_decay_from_stats(
        nucleotide_landscape,
        kernels["e_coli_directed"],
        directed=True,
        squared_norm=squared_norm,
    )
    a_thaliana_weighted_result = analytical_squared_decay_from_stats(
        nucleotide_landscape,
        kernels["a_thaliana_weighted"],
        directed=False,
        squared_norm=squared_norm,
    )
    a_thaliana_directed_result = analytical_squared_decay_from_stats(
        nucleotide_landscape,
        kernels["a_thaliana_directed"],
        directed=True,
        squared_norm=squared_norm,
    )

    uniform_rho = float(uniform_result["rho_2"])
    uniform_denominator = float(uniform_result["denominator"])
    uniform_b_zero = float(uniform_result["b_0"])

    e_coli_bound_17 = weighted_bound_17(
        squared_norm,
        uniform_denominator,
        kernels["e_coli_weighted"],
    )
    e_coli_bound_21 = directed_bound_21(
        squared_norm,
        uniform_denominator,
        uniform_b_zero,
        kernels["e_coli_directed"],
        e_coli_directed_result,
        values_ndim,
    )
    a_thaliana_bound_17 = weighted_bound_17(
        squared_norm,
        uniform_denominator,
        kernels["a_thaliana_weighted"],
    )
    a_thaliana_bound_21 = directed_bound_21(
        squared_norm,
        uniform_denominator,
        uniform_b_zero,
        kernels["a_thaliana_directed"],
        a_thaliana_directed_result,
        values_ndim,
    )

    numeric_rows[r"Uniform ($\rho_2$)"][landscape_name] = uniform_rho
    numeric_rows[r"$\mathit{E.\ coli}$ weighted ($\tilde{\rho}_2$)"][landscape_name] = float(e_coli_weighted_result["rho_2"])
    numeric_rows[r"Bound (17), $\mathit{E.\ coli}$ weighted"][landscape_name] = e_coli_bound_17
    numeric_rows[r"$\mathit{E.\ coli}$ directed ($\bar{\rho}_2$)"][landscape_name] = float(e_coli_directed_result["rho_2"])
    numeric_rows[r"Bound (21), $\mathit{E.\ coli}$ directed"][landscape_name] = e_coli_bound_21
    numeric_rows[r"$\mathit{A.\ thaliana}$ weighted ($\tilde{\rho}_2$)"][landscape_name] = float(a_thaliana_weighted_result["rho_2"])
    numeric_rows[r"Bound (17), $\mathit{A.\ thaliana}$ weighted"][landscape_name] = a_thaliana_bound_17
    numeric_rows[r"$\mathit{A.\ thaliana}$ directed ($\bar{\rho}_2$)"][landscape_name] = float(a_thaliana_directed_result["rho_2"])
    numeric_rows[r"Bound (21), $\mathit{A.\ thaliana}$ directed"][landscape_name] = a_thaliana_bound_21

    diagnostics[landscape_name] = {
        "num_nucleotide_sites": float(values_ndim),
        "num_nucleotide_genotypes": float(nucleotide_landscape.size),
        "e_coli_weighted_abs_difference": abs(uniform_rho - float(e_coli_weighted_result["rho_2"])),
        "e_coli_directed_abs_difference": abs(uniform_rho - float(e_coli_directed_result["rho_2"])),
        "a_thaliana_weighted_abs_difference": abs(uniform_rho - float(a_thaliana_weighted_result["rho_2"])),
        "a_thaliana_directed_abs_difference": abs(uniform_rho - float(a_thaliana_directed_result["rho_2"])),
    }

numeric_table = pd.DataFrame.from_dict(numeric_rows, orient="index")
numeric_table = numeric_table[[name for name, _ in LANDSCAPE_FILES]]
if not np.all(np.isfinite(numeric_table.to_numpy(dtype=float))):
    raise FloatingPointError("The table contains non-finite values.")
if np.any(numeric_table.to_numpy(dtype=float) < -1e-12):
    raise ValueError("The table contains negative values.")

for landscape_name in numeric_table.columns:
    checks = (
        (r"Bound (17), $\mathit{E.\ coli}$ weighted", "e_coli_weighted_abs_difference"),
        (r"Bound (21), $\mathit{E.\ coli}$ directed", "e_coli_directed_abs_difference"),
        (r"Bound (17), $\mathit{A.\ thaliana}$ weighted", "a_thaliana_weighted_abs_difference"),
        (r"Bound (21), $\mathit{A.\ thaliana}$ directed", "a_thaliana_directed_abs_difference"),
    )
    for row_name, diagnostic_name in checks:
        if numeric_table.loc[row_name, landscape_name] + 1e-10 < diagnostics[landscape_name][diagnostic_name]:
            raise AssertionError(f"{row_name} does not bound {diagnostic_name} for {landscape_name}.")

formatted_table = numeric_table.map(lambda value: f"{value:.{DECIMALS}f}")
display(formatted_table)


## Save LaTeX and rendered table

In [ ]:
def dataframe_to_latex_with_hlines(table: pd.DataFrame) -> str:
    """Render the manuscript table with group hlines.

    Parameters:
    - table: pd.DataFrame
        Formatted table with manuscript row labels and landscape columns.

    Returns:
    - str
        LaTeX tabular string.
    """
    lines = [
        r"\begin{tabular}{lrrrr}",
        r"\toprule",
        "Mutation model & " + " & ".join(table.columns) + r" \\",
        r"\midrule",
    ]
    hline_after = {
        r"Bound (17), $\mathit{E.\ coli}$ weighted",
        r"Bound (21), $\mathit{E.\ coli}$ directed",
        r"Bound (17), $\mathit{A.\ thaliana}$ weighted",
    }
    for row_label, row in table.iterrows():
        lines.append(f"{row_label} & " + " & ".join(str(row[column]) for column in table.columns) + r" \\")
        if row_label in hline_after:
            lines.append(r"\midrule")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)


def save_rendered_table(table: pd.DataFrame, stem: str) -> Figure:
    """Save a rendered matplotlib table.

    Parameters:
    - table: pd.DataFrame
        Formatted table to render.
    - stem: str
        Filename stem for saved figure files.

    Returns:
    - Figure
        Rendered matplotlib figure.
    """
    figure_height = 0.48 * (len(table.index) + 2)
    fig, ax = plt.subplots(figsize=(9.0, figure_height), dpi=PANEL_DPI)
    ax.axis("off")
    render_table = ax.table(
        cellText=table.reset_index().to_numpy(),
        colLabels=["Mutation model", *table.columns],
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    render_table.auto_set_font_size(False)
    render_table.set_fontsize(8)
    render_table.scale(1.0, 1.35)
    for (row, column), cell in render_table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
        if column == 0:
            cell.set_text_props(ha="left")
            cell.set_width(0.46)
        else:
            cell.set_width(0.13)
    fig.tight_layout()
    if SAVE_FIGURES:
        for suffix in ("pdf", "png"):
            destination = FIGURES_DIR / suffix
            destination.mkdir(parents=True, exist_ok=True)
            fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches="tight")
    return fig

latex_table = dataframe_to_latex_with_hlines(formatted_table)
TABLE_TEX_PATH.write_text(latex_table + "\n", encoding="utf-8")
with TABLE_PKL_PATH.open("wb") as handle:
    pickle.dump(
        {
            "numeric_table": numeric_table,
            "formatted_table": formatted_table,
            "latex_table": latex_table,
            "diagnostics": diagnostics,
            "metadata": {
                "paper_reference": "Figure 6 mutation-model analytical rho bounds table",
                "eq17_label": "eq:absolute_error_weighted",
                "eq21_label": "eq:absolute_error_directed",
                "landscape_order": [name for name, _ in LANDSCAPE_FILES],
            },
        },
        handle,
    )

print(latex_table)
display(Latex(latex_table))
fig = save_rendered_table(formatted_table, FIGURE_STEM)
plt.show()

print(f"Saved pickle: {TABLE_PKL_PATH}")
print(f"Saved LaTeX: {TABLE_TEX_PATH}")
if SAVE_FIGURES:
    print(f"Saved rendered table: {FIGURES_DIR / 'pdf' / (FIGURE_STEM + '.pdf')}")
    print(f"Saved rendered table: {FIGURES_DIR / 'png' / (FIGURE_STEM + '.png')}")
